In [ ]:
import pandas as pd
import pickle

test = pd.read_pickle("../data/processed/test_data.pkl")

with open("../models/model.pkl", "rb") as f:
    model = pickle.load(f)

In [ ]:
features = [
    'return', 'ma_5', 'ma_10', 'momentum_5',
    'volatility_10', 'volume_norm',
    'return_lag_1', 'return_lag_2', 'return_lag_3',
    'return_lag_4', 'return_lag_5'
]

X_test = test[features]
proba = model.predict_proba(X_test)[:, 1]

In [ ]:
vol = test['volatility_10'].values

def get_signal(p, v):
    if v > 0.04:
        if p > 0.75: return 1
        elif p < 0.25: return -1
        else: return 0
    else:
        if p > 0.6: return 1
        elif p < 0.4: return -1
        else: return 0

signals = [get_signal(p, v) for p, v in zip(proba, vol)]

In [ ]:
cash = 1000
btc = 0
fee = 0.0015

prices = test['Close'].values

for i in range(len(signals)):
    price = prices[i]
    signal = signals[i]
    if signal == 1 and cash > 0:
        btc = (cash / price) * (1 - fee)
        cash = 0
    elif signal == -1 and btc > 0:
        cash = (btc * price) * (1 - fee)
        btc = 0

final_value = cash + btc * prices[-1]
print("Final Value:", final_value)

Final Value: 996.4302602393212


In [ ]:
buy_hold = test['Close'].iloc[-1] / test['Close'].iloc[0]
print("Buy & Hold:", 1000 * buy_hold)

Buy & Hold: 884.0369094443039


In [ ]:
import pickle

results = {
    'final_value': final_value,
    'buy_hold': 1000 * buy_hold,
    'signals': signals,
    'prices': prices
}

with open("../data/processed/backtest_results.pkl", "wb") as f:
    pickle.dump(results, f)

print("backtest_results.pkl saved!")

backtest_results.pkl saved!
